<a href="https://colab.research.google.com/github/PadmanavaParui/btc-forecaster/blob/master/btc_forecaster.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install arch -q

imports + fetching the BTC

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests, json
import scipy.stats as stats
from arch import arch_model
from tqdm import tqdm

# ── Data fetch ───────────────────────────────────────────
def fetch_btc_hourly(n_bars=1000):
    r = requests.get(
        "https://data-api.binance.vision/api/v3/klines",
        params={"symbol": "BTCUSDT", "interval": "1h", "limit": n_bars},
        timeout=10
    )
    r.raise_for_status()
    df = pd.DataFrame(r.json(), columns=[
        "open_time","open","high","low","close","volume",
        "close_time","quote_vol","trades","taker_buy_base","taker_buy_quote","ignore"
    ])
    df["close"]     = df["close"].astype(float)
    df["open_time"] = pd.to_datetime(df["open_time"], unit="ms", utc=True)
    df.set_index("open_time", inplace=True)
    return df["close"].sort_index()

prices = fetch_btc_hourly(1000)
print(f"✓ {len(prices)} bars fetched")
print(f"  From  : {prices.index[0]}")
print(f"  To    : {prices.index[-1]}")
print(f"  Price : ${prices.iloc[-1]:,.2f}")

✓ 1000 bars fetched
  From  : 2026-03-22 14:00:00+00:00
  To    : 2026-05-03 05:00:00+00:00
  Price : $78,243.28


global fit

In [ ]:
np.random.seed(42)
log_ret   = np.log(prices / prices.shift(1)).dropna()
mu_hourly = log_ret.mean()
S0        = prices.iloc[-1]
dt        = 1
n_sims    = 10_000
n_hours   = 1

# ── FIGARCH fit (unchanged from our original) ────────────────────────────
am  = arch_model(log_ret * 100, vol='FIGARCH', p=1, o=0, q=1, dist='studentst')
res = am.fit(disp='off')
sigma_fig = res.conditional_volatility / 100
resid     = (log_ret * 100 - res.params['mu']) / res.conditional_volatility
nu        = max(4, stats.t.fit(resid, floc=0, fscale=1)[0])

# ── Helper functions (copied verbatim from our notebook) ─────────────────
def rolling_entropy(x, window=60, bins=20):
    def ent(v):
        p, _ = np.histogram(v, bins=bins, density=True)
        p = p[p > 0]
        return -np.sum(p * np.log(p))
    return x.rolling(window).apply(ent, raw=True)

H_series   = rolling_entropy(resid)
M_series   = log_ret.abs().rolling(60).mean()
bar_sigma2 = (sigma_fig**2).mean()
redundancy  = 1 + 0.1 * np.log1p(prices.rolling(5).var() / prices.rolling(20).var())
info_filter = (H_series > H_series.mean()).astype(float)
H_max, M_max = H_series.max(), M_series.max()
α0, δ0 = 0.5, 0.3
if α0 * H_max + δ0 * M_max >= 1:
    fac = 0.95 / (α0 * H_max + δ0 * M_max)
    α0 *= fac; δ0 *= fac
base_params = {'alpha': α0, 'delta': δ0, 'gamma': 0.2, 'kappa': 0.1, 'eta': 1e-3}

def update_params(p, sigma2, bar_sigma2, t):
    err = sigma2 - bar_sigma2
    lr  = p['eta'] / (1 + t**0.55)
    p['gamma'] = np.clip(p['gamma'] + lr * err, 0.01, 0.5)
    return p

def simulate_cyber_gbm(S0, mu, sigma_fig, H, M,
                       params, bar_sigma2, n_steps, dt=1, eps=1e-6):
    S = np.zeros(n_steps + 1)
    V = np.zeros(n_steps + 1)
    S[0] = S0
    sigma2 = sigma_fig.iloc[-1] ** 2
    H_max = H.max() if H.max() > 0 else 1.0
    M_max = M.max() if M.max() > 0 else 1.0
    for t in range(1, n_steps + 1):
        current = -1
        H_val = min(H.iloc[current] / H_max, 1.0)
        M_val = min(M.iloc[current] / M_max, 1.0)
        crisis  = (H_val > 0.8) or (M_val > 0.8)
        delta_t = params['delta'] if crisis else 0.0
        sigma2 = (
            sigma_fig.iloc[current]**2 * (1 + params['alpha'] * H_val + delta_t * M_val)
            + params['gamma'] * (bar_sigma2 - sigma2)
        )
        sigma2 *= max(1e-12, redundancy.iloc[current])
        sigma2 *= 1 + 0.5 * info_filter.iloc[current]
        sigma2 = max(eps, min(sigma2, 0.5))
        Z    = np.random.standard_t(nu) * np.sqrt((nu - 2) / nu)
        S[t] = S[t-1] * np.exp((mu - 0.5 * sigma2) * dt + np.sqrt(sigma2 * dt) * Z)
        V[t] = sigma2
        params = update_params(params, sigma2, bar_sigma2, t)
    return S, V

def simulate_mc(S0, mu, sigma_fig, H, M, bar_sigma2, n_sims=10_000, n_days=1):
    out = np.zeros((n_sims, n_days + 1))
    for i in range(n_sims):
        paths, _ = simulate_cyber_gbm(
            S0, mu, sigma_fig, H, M,
            base_params.copy(), bar_sigma2, n_days, dt
        )
        out[i] = paths
    return out

print(f"✓ ν = {nu:.2f} | Current vol = {sigma_fig.iloc[-1]*100:.4f}%/hr")

✓ ν = 10.75 | Current vol = 0.2539%/hr


backtest, evaulate and saving

In [ ]:
def backtest_btc(prices, train=180, test=720):
    """
    Rolls over 720 hourly bars.
    At each step i, only log_ret[:i] is visible — no peeking.

    Index alignment (this is what was broken before):
      log_ret has N-1 rows vs prices N rows
      log_ret.iloc[i] = return that PRODUCED prices.iloc[i+1]
      so: current bar  = prices.iloc[i+1]
          next bar     = prices.iloc[i+2]   ← the thing we're predicting
    """
    log_ret = np.log(prices / prices.shift(1)).dropna()

    # sanity check
    needed = train + test + 2
    assert len(prices) >= needed, f"Need {needed} bars, only have {len(prices)}"

    results = []

    for i in tqdm(range(train, train + test), desc="Backtesting"):
        train_ret = log_ret.iloc[i - train : i]          # past only

        # fit FIGARCH on this window
        try:
            am_bt  = arch_model(train_ret * 100, vol='FIGARCH',
                                p=1, o=0, q=1, dist='studentst')
            res_bt = am_bt.fit(disp='off')
        except Exception:
            # if FIGARCH fails to converge, skip this bar
            continue

        sigma_bt  = res_bt.conditional_volatility / 100
        resid_bt  = (train_ret * 100 - res_bt.params['mu']) / res_bt.conditional_volatility
        nu_bt     = max(4, stats.t.fit(resid_bt, floc=0, fscale=1)[0])

        H_bt      = rolling_entropy(resid_bt)
        M_bt      = train_ret.abs().rolling(60).mean()
        bar_s2_bt = (sigma_bt**2).mean()

        # redundancy & info_filter for this window (computed on prices slice)
        prices_slice  = prices.iloc[i - train : i + 1]
        red_bt  = 1 + 0.1 * np.log1p(prices_slice.rolling(5).var() / prices_slice.rolling(20).var())
        info_bt = (H_bt > H_bt.mean()).astype(float)

        # correct index alignment — no off-by-one
        S0_bt  = float(prices.iloc[i + 1])
        actual = float(prices.iloc[i + 2])

        # temporarily override globals for simulate_cyber_gbm
        global redundancy, info_filter
        _red_save, _inf_save = redundancy, info_filter
        redundancy, info_filter = red_bt, info_bt

        paths_bt = simulate_mc(S0_bt, float(train_ret.mean()),
                               sigma_bt, H_bt, M_bt, bar_s2_bt,
                               n_sims=10_000, n_days=1)

        redundancy, info_filter = _red_save, _inf_save  # restore

        S_t1          = paths_bt[:, 1]
        low95, high95 = np.percentile(S_t1, [2.5, 97.5])
        width = high95 - low95
        hit   = bool(low95 <= actual <= high95)

        alpha = 0.05
        if actual < low95:
            winkler = width + (2/alpha) * (low95 - actual)
        elif actual > high95:
            winkler = width + (2/alpha) * (actual - high95)
        else:
            winkler = width

        results.append({
            "bar_index"    : i - train,
            "timestamp"    : prices.index[i + 1].isoformat(),
            "current_price": round(S0_bt, 2),
            "lower_95"     : round(float(low95), 2),
            "upper_95"     : round(float(high95), 2),
            "actual"       : round(actual, 2),
            "hit"          : hit,
            "width"        : round(width, 2),
            "winkler"      : round(float(winkler), 2)
        })

    return results

# ── Run ───────────────────────────────────────────────────────────────────
results = backtest_btc(prices, train=180, test=720)

# ── Score ─────────────────────────────────────────────────────────────────
hits     = [p["hit"]     for p in results]
widths   = [p["width"]   for p in results]
winklers = [p["winkler"] for p in results]

coverage     = np.mean(hits)
avg_width    = np.mean(widths)
mean_winkler = np.mean(winklers)

print("=" * 45)
print(f"  Predictions : {len(results)}")
print(f"  Coverage    : {coverage:.4f}   (target ~0.95)")
print(f"  Avg width   : ${avg_width:,.2f}")
print(f"  Winkler     : ${mean_winkler:,.2f}")
print("=" * 45)

# ── Save ──────────────────────────────────────────────────────────────────
with open("backtest_results.jsonl", "w") as f:
    for p in results:
        f.write(json.dumps(p) + "\n")
print("✓ Saved backtest_results.jsonl")

from google.colab import files
files.download("backtest_results.jsonl")

Backtesting: 100%|██████████| 720/720 [16:25<00:00,  1.37s/it]

  Predictions : 720
  Coverage    : 0.9750   (target ~0.95)
  Avg width   : $1,552.62
  Winkler     : $1,858.42
✓ Saved backtest_results.jsonl


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>